[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/dynamic-breakpoints.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239526-lesson-4-dynamic-breakpoints)

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Dynamic breakpoints

**一句话定位**：静态 breakpoint 是 compile 时写死 `interrupt_before=[...]`；`interrupt()` 让 graph 能**在运行时按逻辑自己决定要不要暂停**，并拿到人的回复。

**目标**：动态地自我打断（internal breakpoint）有两个好处：

1. 可以**有条件地**触发（节点内部按自定义逻辑判断）
2. 可以**告诉用户为什么被打断**（`interrupt()` 里传任意 payload），并把人的回复直接拿回来

下面构造一个按输入长度判断是否 `interrupt()` 的 graph：超过 5 个字符就暂停，等人给一个更短的输入。

</div>

In [ ]:
from IPython.display import Image, display

from typing_extensions import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph
from langgraph.types import interrupt, Command

class State(TypedDict):
    input: str

def step_1(state: State) -> State:
    print("---Step 1---")
    return state

def step_2(state: State) -> State:
    if len(state["input"]) > 5:
        # 暂停，把说明抛给调用方；人 resume 的值就是返回值，直接写回 state
        new_input = interrupt(f"Input too long ({state['input']}), give me a shorter one:")
        return {"input": new_input}
    print("---Step 2---")
    return state

def step_3(state: State) -> State:
    print("---Step 3---")
    return state

builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

# Set up memory
memory = InMemorySaver()

# Compile the graph with memory
graph = builder.compile(checkpointer=memory)

# View
display(Image(graph.get_graph().draw_mermaid_png()))

用一个超过 5 个字符的输入来跑一下这个 graph。

In [ ]:
initial_input = {"input": "hello world"}
thread_config = {"configurable": {"thread_id": "1"}}

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread_config, stream_mode="values"):
    print(event)

此时查看 graph state，会看到下一个待执行节点是 `step_2`。

In [ ]:
state = graph.get_state(thread_config)
print(state.next)

可以看到 `Interrupt` 被记录在了 state 里。

In [ ]:
print(state.tasks)

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**续跑规则**：`Command(resume=值)` 之后，`step_2` **从头重跑**，这一次 `interrupt()` 直接返回 resume 的值而不再暂停。若误用 `invoke(None)` 续跑，不带新值，条件（长度 > 5）依旧满足，会再次暂停。

</div>

In [ ]:
for event in graph.stream(Command(resume="hi"), thread_config, stream_mode="values"):
    print(event)

In [ ]:
state = graph.get_state(thread_config)
print(state.next)

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

### 通过 LangGraph API 使用

**🆕 2026 更新**：

Studio 已经可以本地运行、通过浏览器访问，改名为 **LangSmith Studio**（取代视频里的旧版 Desktop App）。配置参考课程开头 "Getting Setup" 指南，介绍见[这里](https://docs.langchain.com/langsmith/studio)，本地部署见[这里](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)。

在本模块 `/studio` 目录下终端运行：

```
langgraph dev
```

应看到：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器访问上面的 **Studio UI** 链接。

</div>

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

用 SDK 连接到这个本地服务。

In [ ]:
from langgraph_sdk import get_client

# This is the URL of the local development server
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# Search all hosted graphs
assistants = await client.assistants.search()

In [ ]:
thread = await client.threads.create()
input_dict = {"input": "hello world"}

async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="dynamic_breakpoints",
    input=input_dict,
    stream_mode="values",):
    
    print(f"Receiving new event of type: {chunk.event}...")
    print(chunk.data)
    print("\n\n")

In [ ]:
current_state = await client.threads.get_state(thread['thread_id'])

In [ ]:
current_state['next']

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="dynamic_breakpoints",
    command={"resume": "hi!"},        # 续跑：resume 的值就是 interrupt() 的返回值
    stream_mode="values",):

    print(f"Receiving new event of type: {chunk.event}...")
    print(chunk.data)
    print("\n\n")

In [ ]:
current_state = await client.threads.get_state(thread['thread_id'])
current_state